# Part 2 — mtcars Linear Regression

We replicate the R analysis `lm(mpg ~ ., data = mtcars)` using **statsmodels** OLS, then answer the exam questions and run a forward stepwise model selection by AIC.

## Cell 1 — Load data

In [ ]:
import pandas as pd

# First column is the car name — use it as index
mtcars = pd.read_csv('../mtcars.csv', index_col=0)

print('Shape:', mtcars.shape)
print('Columns:', list(mtcars.columns))
mtcars.head()

## Cell 2 — Full OLS model: mpg ~ all predictors

Equivalent to `lm(mpg ~ ., data = mtcars)` followed by `summary(M)` in R.

In [ ]:
import statsmodels.api as sm
import numpy as np

y = mtcars['mpg']
X = mtcars.drop(columns='mpg')

# Add intercept (statsmodels does not include it by default)
X_const = sm.add_constant(X)

model   = sm.OLS(y, X_const)
results = model.fit()

print(results.summary())

**Q1.** Which predictors seem to have an effect on mpg? Make your criterion explicit.


## Cell 3 — Q1: Which predictors have a significant effect on mpg? (p-value < 0.05)

In [ ]:
pvalues = results.pvalues.drop('const')   # exclude intercept
significant = pvalues[pvalues < 0.05]

print('Predictors significant at the 5% level (p < 0.05):')
if len(significant) == 0:
    print('  None — when all predictors are included simultaneously, multicollinearity')
    print('  inflates standard errors, masking individual significance.')
else:
    for var, pval in significant.items():
        print(f'  {var:12s}  p = {pval:.4f}')

print()
print('All p-values (sorted):')
print(pvalues.sort_values().to_string())

**Q2.** Provide a 95% confidence interval for the `carb` parameter. What can you deduce from the fact that 0 is in the confidence interval? Provide descriptions and formulas to explain what the quantity in column `Pr(>|t|)` represents.


## Cell 4 — Q2: 95% confidence interval for the `carb` coefficient

In [ ]:
ci = results.conf_int(alpha=0.05)   # 95% CI

carb_ci   = ci.loc['carb']
carb_coef = results.params['carb']
carb_pval = results.pvalues['carb']

print(f'carb coefficient : {carb_coef:.4f}')
print(f'95% CI           : [{carb_ci[0]:.4f}, {carb_ci[1]:.4f}]')
print(f'p-value          : {carb_pval:.4f}')
print()
print('Full 95% CI table:')
print(ci.to_string())

**Q3.** Provide descriptions and formulas to explain what `p-value: 3.793e-07` represents. What to conclude from this value? Does it seem in discordance with your answer in Q1? What could explain such discordance?


**Q4.** What strategy could be used to obtain a linear regression model with a maximal number of predictors, all of which have significant effect at level 5%? Can you run it in practice? If yes, provide the estimates. If not, provide the reason.


## Cell 5 — Explanation: p-values, the F-test, and apparent discordance

### Q2 continued — What does 0 inside the CI for `carb` mean?

If the 95% confidence interval for the `carb` coefficient **contains 0**, we cannot reject at the 5% level the hypothesis that `carb` has no linear effect on `mpg` **given all other predictors are in the model**. In other words, we cannot conclude that the number of carburetors has a statistically significant partial effect on fuel efficiency once the other 10 variables are controlled for.

### What is Pr(>|t|) (the individual p-value)?

`Pr(>|t|)` is the p-value of the **t-test** for a single coefficient.  
- The null hypothesis is H₀: βⱼ = 0 (predictor j has no effect, conditional on all others).  
- The test statistic is t = β̂ⱼ / SE(β̂ⱼ), which under H₀ follows a t-distribution with n − p − 1 degrees of freedom.  
- A small p-value (< 0.05) indicates that the coefficient is significantly different from zero in the presence of the other predictors.

### Q3 — The overall F-test p-value (≈ 3.79 × 10⁻⁷)

The F-test evaluates a **joint null hypothesis**: H₀: β₁ = β₂ = … = β₁₀ = 0 (none of the predictors explain mpg).  
A very small F-test p-value (3.79 × 10⁻⁷) tells us the model as a whole explains a highly significant portion of the variance in `mpg` — **at least one** predictor is useful.

**Apparent discordance with Q1:** It is common to find that the overall F-test is highly significant even though no individual t-test reaches significance at 5%. This happens because of **multicollinearity**: the predictors are strongly intercorrelated (e.g., `cyl`, `disp`, `hp`, and `wt` all measure engine size/power). Multicollinearity inflates the standard errors of individual coefficients, reducing each t-statistic without affecting the F-statistic (which tests them jointly). The signal is real, but it is shared across correlated predictors in a way that no single one can claim individual credit.

### Q4 — Strategy to find a model with all significant predictors at 5%

Two standard strategies:  
1. **Backward elimination**: start with the full model; at each step, remove the predictor with the highest (non-significant) p-value; refit; repeat until all remaining predictors are significant.  
2. **Forward stepwise by AIC/BIC**: start with the intercept-only model; at each step, add the predictor that most reduces AIC (or BIC); stop when no addition improves the criterion. AIC penalises model complexity and tends to keep predictors that genuinely improve predictive fit.

The AIC-based approach is preferred because it selects based on predictive performance rather than arbitrary significance thresholds, and it naturally avoids the multiple-testing inflation inherent in repeated p-value checking.

**Q5.** Describe and run some algorithm to find the best model in some sense.


## Cell 6 — Q5: Forward stepwise selection by AIC

**Algorithm:**
1. Start with the intercept-only model.  
2. For each predictor not yet in the model, fit the model with that predictor added and record its AIC.  
3. Add the predictor that gives the **lowest AIC** — but only if it is lower than the current model's AIC.  
4. Repeat steps 2–3 until no addition reduces AIC.  
5. The remaining set of predictors defines the best model.

In [ ]:
def forward_stepwise_aic(y, X):
    """
    Forward stepwise selection based on AIC.
    Returns the ordered list of selected predictor names.
    """
    remaining  = list(X.columns)
    selected   = []
    history    = []

    # Intercept-only model AIC
    intercept_only = sm.OLS(y, sm.add_constant(pd.Series(np.ones(len(y)), index=y.index, name='const'))).fit()
    current_aic    = intercept_only.aic
    print(f'Step 0 (intercept only) — AIC: {current_aic:.3f}')

    step = 1
    while remaining:
        best_aic  = current_aic
        best_pred = None

        for pred in remaining:
            candidates    = selected + [pred]
            X_cand        = sm.add_constant(X[candidates])
            fit           = sm.OLS(y, X_cand).fit()
            if fit.aic < best_aic:
                best_aic  = fit.aic
                best_pred = pred

        if best_pred is None:
            print(f'Step {step}: no improvement — stopping.')
            break

        selected.append(best_pred)
        remaining.remove(best_pred)
        current_aic = best_aic
        history.append((step, best_pred, best_aic))
        print(f'Step {step}: add "{best_pred}" — AIC: {best_aic:.3f}  (selected so far: {selected})')
        step += 1

    return selected, history


selected_vars, selection_history = forward_stepwise_aic(y, X)
print()
print('Final selected predictors:', selected_vars)

## Cell 7 — Final model summary

In [ ]:
X_final = sm.add_constant(X[selected_vars])
model_final   = sm.OLS(y, X_final)
results_final = model_final.fit()

print('=== Final model after forward stepwise AIC selection ===')
print(f'Predictors: {selected_vars}')
print()
print(results_final.summary())

# Show which predictors are now significant
final_pvals = results_final.pvalues.drop('const')
sig_final   = final_pvals[final_pvals < 0.05]
print()
print('Predictors significant at 5% in the final model:')
for var, pval in sig_final.items():
    print(f'  {var:12s}  p = {pval:.4f}')